
# Busqueda A* para el problema del 8-Puzzle (heurística propia)

Este cuaderno implementa el algoritmo de busqueda **A\*** (A estrella) para resolver
el problema del **8-puzzle**, usando como una **heurística basada en filas y columnas**.

**Entrada:** dos archivos de texto ubicados en el mismo directorio que este cuaderno:

- `estadoinicial.txt`: contiene el tablero de inicio (3 lineas de 3 digitos cada una).
- `estadofinal.txt`: contiene el tablero meta (3 lineas de 3 digitos cada una).

El espacio vacio se representa con el digito `0`.

Ejemplo `estadoinicial.txt`:
```
326
504
187
```

Ejemplo `estadofinal.txt`:
```
847
265
310
```

**Contenido del cuaderno**

1. Importacion de librerias
2. Lectura de los archivos de entrada
3. Representacion del estado y generacion de vecinos (movimientos validos)
4. **Funcion heuristica (Explicacion)**
5. Verificacion de solubilidad (paridad de inversiones)
6. Algoritmo A\*
7. Ejecucion y presentacion de la solucion paso a paso


## 1. Importacion de librerias

In [36]:
import heapq
import itertools
import os


## 2. Lectura de los archivos de entrada

El tablero se representa internamente como una **tupla de tuplas** (3x3), porque
las tuplas son inmutables y por lo tanto pueden usarse como llaves de diccionario
y guardarse en conjuntos (`set`) -- algo indispensable para llevar el registro de
estados visitados en la busqueda.


In [37]:
def leer_estado(ruta_archivo):
    """
    Lee un archivo de texto con 3 lineas de 3 digitos (0-8) y devuelve
    el tablero como una tupla de tuplas, ej:
    ((3,2,6),(5,0,4),(1,8,7))
    """
    with open(ruta_archivo, 'r') as f:
        lineas = [linea.strip() for linea in f if linea.strip() != '']

    if len(lineas) != 3:
        raise ValueError(f"El archivo {ruta_archivo} debe tener exactamente 3 lineas no vacias.")

    tablero = []
    for linea in lineas:
        fila = [int(c) for c in linea]
        if len(fila) != 3:
            raise ValueError(f"Cada linea de {ruta_archivo} debe tener 3 digitos.")
        tablero.append(tuple(fila))

    return tuple(tablero)


def imprimir_tablero(estado):
    """Imprime el tablero de forma legible, mostrando '_' para el espacio vacio."""
    for fila in estado:
        print(' '.join(str(v) if v != 0 else '_' for v in fila))
    print()


## 3. Representacion del estado y generacion de vecinos

Dado un estado, se busca la posicion del `0` (espacio vacio) y se generan los
estados sucesores validos moviendo la ficha adyacente (arriba, abajo, izquierda,
derecha) hacia esa posicion vacia.


In [38]:
def encontrar_blanco(estado):
    """Devuelve (fila, columna) de la posicion del 0 en el tablero."""
    for i in range(3):
        for j in range(3):
            if estado[i][j] == 0:
                return i, j
    raise ValueError("El estado no contiene un espacio vacio (0).")


def generar_vecinos(estado):
    """
    Genera los estados vecinos validos a partir de 'estado', moviendo el
    espacio vacio en las 4 direcciones posibles (cuando aplica).
    Devuelve una lista de tuplas (nuevo_estado, nombre_del_movimiento).
    """
    vecinos = []
    fila_b, col_b = encontrar_blanco(estado)

    # (delta_fila, delta_columna, nombre_del_movimiento)
    movimientos = [(-1, 0, 'Arriba'), (1, 0, 'Abajo'),
                   (0, -1, 'Izquierda'), (0, 1, 'Derecha')]

    for df, dc, nombre in movimientos:
        nf, nc = fila_b + df, col_b + dc
        if 0 <= nf < 3 and 0 <= nc < 3:
            tablero_mutable = [list(fila) for fila in estado]
            valor_actual = tablero_mutable[fila_b][col_b]
            tablero_mutable[fila_b][col_b] = tablero_mutable[nf][nc]
            tablero_mutable[nf][nc] = valor_actual
            nuevo_estado = tuple(tuple(fila) for fila in tablero_mutable)
            vecinos.append((nuevo_estado, nombre))

    return vecinos

## 4. Función heurística propuesta

La heurística propuesta consiste en revisar cada ficha (excepto el espacio vacío) y comparar su posición con la del estado objetivo.

Para cada ficha:

- Si no está en la **fila** correcta, suma **1**.
- Si no está en la **columna** correcta, suma **1**.

Finalmente, la heurística es la suma de esos valores para todas las fichas.

Cada ficha puede aportar:

- 0: está en la posición correcta.
- 1: coincide solo la fila o la columna.
- 2: no coincide ni la fila ni la columna.

Esta heurística estima de forma sencilla qué tan desordenado está el tablero y prioriza los estados donde más fichas ya están alineadas con el estado objetivo.


In [39]:
def construir_posiciones_meta(estado_final):
    """
    Construye un diccionario que mapea cada ficha a su posición en el estado final.
    """
    posiciones = {}
    for i in range(3):
        for j in range(3):
            posiciones[estado_final[i][j]] = (i, j)
    return posiciones

def heuristica_filas_columnas(estado, posiciones_meta):
    """
    Calcula la heurística de filas y columnas para un estado dado.
    """
    h = 0
    for i in range(3):
        for j in range(3):
            ficha = estado[i][j]
            if ficha != 0:
                fila_meta, columna_meta = posiciones_meta[ficha]
                if i != fila_meta:
                    h += 1
                if j != columna_meta:
                    h += 1
    return h



## 5. Verificacion de solubilidad

No todos los pares (inicial, meta) del 8-puzzle son alcanzables entre si. Para un
tablero de ancho impar (3x3) el criterio es:

> El estado meta es alcanzable desde el estado inicial **si y solo si** ambos
> estados tienen la **misma paridad** en su numero de inversiones (contando las
> fichas en orden de lectura, ignorando el espacio vacio).

Verificar esto antes de ejecutar A\* evita una busqueda que recorreria **todo**
el espacio de estados alcanzable sin encontrar nunca la meta.


In [40]:
def contar_inversiones(secuencia):
    """
    Cuenta el numero de inversiones de una secuencia de numeros
    (una inversion es un par (i, j) con i < j pero secuencia[i] > secuencia[j]).
    """
    inversiones = 0
    n = len(secuencia)
    for i in range(n):
        for j in range(i + 1, n):
            if secuencia[i] > secuencia[j]:
                inversiones += 1
    return inversiones


def es_soluble(estado_inicial, estado_final):
    """
    Determina si estado_final es alcanzable desde estado_inicial,
    comparando la paridad del numero de inversiones de ambos estados
    (valido para tableros de ancho impar, como el 8-puzzle de 3x3).
    """
    plano_inicial = [v for fila in estado_inicial for v in fila if v != 0]
    plano_final = [v for fila in estado_final for v in fila if v != 0]

    inv_inicial = contar_inversiones(plano_inicial)
    inv_final = contar_inversiones(plano_final)

    return (inv_inicial % 2) == (inv_final % 2)


## 6. Algoritmo A\*

Estructuras utilizadas:

- **Frontera (open list)**: una **cola de prioridad** (`heapq`) ordenada por
  `f(n) = g(n) + h(n)`.
  - `g(n)`: costo real acumulado (numero de movimientos) desde el inicio hasta `n`.
  - `h(n)`: valor heuristico (heurística propuesta) desde `n` hasta la meta.
- **`costo_g`**: diccionario `estado -> mejor g(n) encontrado hasta ahora`.
- **`padres`**: diccionario `estado -> estado padre`, para reconstruir el camino
  solucion al final.
- **`visitados`**: conjunto de estados ya expandidos (closed list), para no
  reprocesarlos.

Como las tuplas de tuplas no tienen un orden total definido de forma util para
desempatar en el heap, se usa un **contador** (`itertools.count`) como segundo
criterio de comparacion, para que `heapq` nunca intente comparar dos tableros
entre si directamente.


In [43]:
def reconstruir_camino(estado_final, padres, movimientos):
    """
    Reconstruye la secuencia de estados (y movimientos) desde el estado
    inicial hasta 'estado_final', siguiendo los punteros a los padres.
    """
    camino = []
    actual = estado_final
    while actual is not None:
        camino.append((actual, movimientos[actual]))
        actual = padres[actual]
    camino.reverse()
    return camino


def a_estrella(estado_inicial, estado_final, funcion_heuristica=heuristica_filas_columnas):
    """
    Ejecuta el algoritmo A* para encontrar el camino de costo minimo
    entre estado_inicial y estado_final.

    Devuelve una tupla (camino, nodos_expandidos, costo_total):
      - camino: lista de (estado, movimiento_realizado) desde el inicio hasta la meta,
                o None si no se encontro solucion.
      - nodos_expandidos: numero de estados que fueron sacados de la frontera y expandidos.
      - costo_total: numero de movimientos de la solucion (o None si no hay solucion).
    """
    posiciones_meta = construir_posiciones_meta(estado_final)

    contador = itertools.count()  # desempate estable para el heap

    costo_g = {estado_inicial: 0}
    padres = {estado_inicial: None}
    movimientos_realizados = {estado_inicial: None}
    visitados = set()

    h_inicial = funcion_heuristica(estado_inicial, posiciones_meta)
    frontera = [(h_inicial, next(contador), estado_inicial)]

    nodos_expandidos = 0

    while frontera:
        f_actual, _, estado_actual = heapq.heappop(frontera)

        if estado_actual in visitados:
            continue
        visitados.add(estado_actual)
        nodos_expandidos += 1

        """
        print(f"\nExpansión {nodos_expandidos}")
        print(f"g = {costo_g[estado_actual]}")
        print(f"h = {funcion_heuristica(estado_actual, posiciones_meta)}")
        print(f"f = {costo_g[estado_actual] + funcion_heuristica(estado_actual, posiciones_meta)}")
        print("Estado actual:")

        for fila in estado_actual:
            print(*fila)*//
        """

        if estado_actual == estado_final:
            camino = reconstruir_camino(estado_actual, padres, movimientos_realizados)
            return camino, nodos_expandidos, costo_g[estado_actual]

        for vecino, nombre_movimiento in generar_vecinos(estado_actual):
            nuevo_costo_g = costo_g[estado_actual] + 1

            if vecino not in costo_g or nuevo_costo_g < costo_g[vecino]:
                costo_g[vecino] = nuevo_costo_g
                h = funcion_heuristica(vecino, posiciones_meta)
                f = nuevo_costo_g + h
                heapq.heappush(frontera, (f, next(contador), vecino))
                padres[vecino] = estado_actual
                movimientos_realizados[vecino] = nombre_movimiento

    # Si la frontera se vacia sin encontrar la meta, no hay solucion
    return None, nodos_expandidos, None


## 7. Ejecucion principal

Lee `estadoinicial.txt` y `estadofinal.txt`, comprueba la solubilidad y ejecuta
A\*, mostrando la solucion paso a paso junto con estadisticas de la busqueda.


In [44]:
ARCHIVO_INICIAL = "estadoinicial.txt"
ARCHIVO_FINAL = "estadofinal.txt"

estado_inicial = leer_estado(ARCHIVO_INICIAL)
estado_final = leer_estado(ARCHIVO_FINAL)

print("Estado inicial:")
imprimir_tablero(estado_inicial)

print("Estado final (meta):")
imprimir_tablero(estado_final)

if not es_soluble(estado_inicial, estado_final):
    print("No existe solucion: el estado final NO es alcanzable desde el estado inicial "
          "(paridad de inversiones distinta).")
else:
    camino, nodos_expandidos, costo_total = a_estrella(
        estado_inicial, estado_final, funcion_heuristica=heuristica_filas_columnas
    )

    if camino is None:
        print("No se encontro solucion (esto no deberia ocurrir si es_soluble devolvio True).")
    else:
        print(f"Solucion encontrada en {costo_total} movimientos")
        print(f"Nodos expandidos/iteraciones: {nodos_expandidos}")
        print()

        for paso, (estado, movimiento) in enumerate(camino):
            if movimiento is None:
                print(f"Paso {paso} (estado inicial):")
            else:
                print(f"Paso {paso} - Movimiento del espacio vacio: {movimiento}")
            imprimir_tablero(estado)

Estado inicial:
3 2 6
5 _ 4
1 8 7

Estado final (meta):
8 4 7
2 6 5
3 1 _

Solucion encontrada en 20 movimientos
Nodos expandidos/iteraciones: 1023

Paso 0 (estado inicial):
3 2 6
5 _ 4
1 8 7

Paso 1 - Movimiento del espacio vacio: Abajo
3 2 6
5 8 4
1 _ 7

Paso 2 - Movimiento del espacio vacio: Izquierda
3 2 6
5 8 4
_ 1 7

Paso 3 - Movimiento del espacio vacio: Arriba
3 2 6
_ 8 4
5 1 7

Paso 4 - Movimiento del espacio vacio: Arriba
_ 2 6
3 8 4
5 1 7

Paso 5 - Movimiento del espacio vacio: Derecha
2 _ 6
3 8 4
5 1 7

Paso 6 - Movimiento del espacio vacio: Abajo
2 8 6
3 _ 4
5 1 7

Paso 7 - Movimiento del espacio vacio: Abajo
2 8 6
3 1 4
5 _ 7

Paso 8 - Movimiento del espacio vacio: Izquierda
2 8 6
3 1 4
_ 5 7

Paso 9 - Movimiento del espacio vacio: Arriba
2 8 6
_ 1 4
3 5 7

Paso 10 - Movimiento del espacio vacio: Arriba
_ 8 6
2 1 4
3 5 7

Paso 11 - Movimiento del espacio vacio: Derecha
8 _ 6
2 1 4
3 5 7

Paso 12 - Movimiento del espacio vacio: Derecha
8 6 _
2 1 4
3 5 7

Paso 13 - Movimien